# 04 - Data Cleaning

## Objective

This notebook cleans, standardizes, and validates the 2025 Airline On-Time Performance dataset before statistical analysis, feature engineering, and machine learning.

The data-cleaning process includes:

- Loading the project configuration and managed raw Delta table
- Validating the approved 32-column schema
- Standardizing dates, binary indicators, numerical fields, and categorical text
- Verifying date parsing and converted data types
- Detecting exact and business-key duplicate records
- Measuring and investigating missing values
- Identifying structurally valid null values associated with cancellations and diversions
- Removing incomplete records that cannot support supervised learning
- Validating permitted categorical and binary value domains
- Checking operational relationships between related variables
- Saving the validated dataset as the managed Delta table `flights_clean`

The resulting table serves as the input for statistical analysis and feature engineering.


#### Load project configuration

The raw dataset is read from the managed Unity Catalog table created during data ingestion. The cleaned dataset is written in Delta format to the processed layer of the project Volume.


In [0]:
# Load the project configuration

from __future__ import annotations

from config import project_config as cfg
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T

print("Project configuration loaded successfully.")
print(f"Source table: {cfg.RAW_TABLE}")
print(f"Clean table: {cfg.CLEAN_TABLE}")
print(f"Processed layer: {cfg.PROCESSED_PATH}")
print(f"Clean Delta output: {cfg.CLEAN_DELTA_PATH}")
print(f"Prediction target: {cfg.TARGET_COLUMN}")


#### Load the raw dataset

The cleaning process begins by loading the `flights_raw` managed Delta table. This table already combines the twelve monthly BTS files for January through December 2025, so the source CSV files do not need to be read again.


In [0]:
# Load the raw flight table

def require_table(table_name: str) -> None:
    """Validate that a required Unity Catalog table exists."""
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table '{table_name}' was not found. "
            "Run the data-ingestion notebook before continuing."
        )


require_table(cfg.RAW_TABLE)

df_raw: DataFrame = spark.table(cfg.RAW_TABLE)

raw_row_count = df_raw.count()
raw_column_count = len(df_raw.columns)

print("Raw dataset loaded successfully.")
print(f"Total records: {raw_row_count:,}")
print(f"Total columns: {raw_column_count}")


In [0]:
# Preview raw dataset

display(df_raw.limit(10))


#### Schema validation

The raw dataset is compared with the approved list of 32 BTS attributes. This validation identifies missing, unexpected, or incorrectly named columns before any cleaning rules are applied.


In [0]:
# Validate approved BTS schema

actual_columns = df_raw.columns

missing_columns = sorted(set(cfg.EXPECTED_RAW_COLUMNS) - set(actual_columns))
unexpected_columns = sorted(set(actual_columns) - set(cfg.EXPECTED_RAW_COLUMNS))

print(f"Expected columns: {len(cfg.EXPECTED_RAW_COLUMNS)}")
print(f"Actual columns: {len(actual_columns)}")

if missing_columns:
    raise ValueError(
        "Schema validation failed. Missing required columns: "
        f"{missing_columns}"
    )

print("All required columns are present.")

if unexpected_columns:
    print(f"Unexpected columns detected: {unexpected_columns}")
else:
    print("No unexpected columns detected.")


In [0]:
# Review raw schema

schema_rows = [
    (field.name, field.dataType.simpleString(), field.nullable)
    for field in df_raw.schema.fields
]

schema_df = spark.createDataFrame(
    schema_rows,
    ["COLUMN_NAME", "CURRENT_DATA_TYPE", "NULLABLE"],
)

display(schema_df)


#### Data type and text standardization

Several fields were inferred using generic data types during ingestion. The following standardizations are applied:

- `FL_DATE` is converted from a string containing date and time into Spark `DateType`.
- Binary indicator fields are converted to integers.
- Code and location fields are trimmed and standardized.
- Empty categorical values are converted to null.
- Numerical performance variables remain numeric.

The raw dataset is not modified, and no records are removed during this stage.


In [0]:
# Review cleaning column groups from project configuration

print("Binary indicator columns:")
for column_name in cfg.BINARY_INDICATOR_COLUMNS:
    print(f"- {column_name}")

print("Code columns:")
for column_name in cfg.CODE_COLUMNS:
    print(f"- {column_name}")


In [0]:
# Standardize data types and text fields

df_clean = df_raw

df_clean = df_clean.withColumn(
    cfg.FLIGHT_DATE_COLUMN,
    F.to_date(
        F.try_to_timestamp(
            F.trim(F.col(cfg.FLIGHT_DATE_COLUMN)),
            F.lit(cfg.FL_DATE_PARSE_FORMAT),
        )
    ),
)

for column_name in cfg.BINARY_INDICATOR_COLUMNS:
    df_clean = df_clean.withColumn(
        column_name,
        F.col(column_name).cast(T.IntegerType()),
    )

for column_name in cfg.INTEGER_COLUMNS:
    df_clean = df_clean.withColumn(
        column_name,
        F.col(column_name).cast(T.IntegerType()),
    )

for column_name in cfg.DOUBLE_COLUMNS:
    df_clean = df_clean.withColumn(
        column_name,
        F.col(column_name).cast(T.DoubleType()),
    )

for column_name in cfg.CODE_COLUMNS:
    cleaned_value = F.upper(F.trim(F.col(column_name)))
    df_clean = df_clean.withColumn(
        column_name,
        F.when(
            cleaned_value.isNull() | (cleaned_value == ""),
            F.lit(None),
        ).otherwise(cleaned_value),
    )

for column_name in cfg.TEXT_COLUMNS:
    cleaned_value = F.trim(F.col(column_name))
    df_clean = df_clean.withColumn(
        column_name,
        F.when(
            cleaned_value.isNull() | (cleaned_value == ""),
            F.lit(None),
        ).otherwise(cleaned_value),
    )

print("Initial type and text standardization completed.")


#### Date parsing validation

The `FL_DATE` column is converted from a string to a Spark date during the data type standardization stage. This validation checks whether any original non-null date values failed to convert successfully.

Review `DATE_PARSE_FAILURES` in the output below. A value of **0** confirms that all available flight dates were parsed successfully.


In [0]:
# Validate FL_DATE parsing

date_validation = (
    df_raw
    .select(
        F.count("*").alias("TOTAL_ROWS"),
        F.sum(
            F.when(F.col(cfg.FLIGHT_DATE_COLUMN).isNull(), 1).otherwise(0)
        ).alias("ORIGINAL_NULL_DATES"),
        F.sum(
            F.when(
                F.col(cfg.FLIGHT_DATE_COLUMN).isNotNull()
                & F.try_to_timestamp(
                    F.trim(F.col(cfg.FLIGHT_DATE_COLUMN)),
                    F.lit(cfg.FL_DATE_PARSE_FORMAT),
                ).isNull(),
                1,
            ).otherwise(0)
        ).alias("DATE_PARSE_FAILURES"),
    )
)

display(date_validation)


#### Schema verification

After applying the initial data type standardization, the schema is verified to confirm that the selected columns have been converted to their intended data types.


In [0]:
# Review standardized column types

converted_schema_rows = [
    (
        field.name,
        field.dataType.simpleString(),
        field.nullable,
    )
    for field in df_clean.select(*cfg.STANDARDIZED_TYPE_COLUMNS).schema.fields
]

converted_schema_df = spark.createDataFrame(
    converted_schema_rows,
    ["COLUMN_NAME", "STANDARDIZED_DATA_TYPE", "NULLABLE"],
)

display(converted_schema_df)


#### Standardized dataset preview

A sample of the standardized dataset is displayed below to verify that the applied transformations were successful.


In [0]:
# Preview standardized dataset

display(
    df_clean.select(*cfg.CLEANING_PREVIEW_COLUMNS).limit(20)
)


#### Duplicate detection

Duplicate records can distort descriptive statistics, bias model training, and produce inaccurate operational insights. Two types of duplicates are evaluated:

- **Exact duplicates** — records in which all variables contain identical values.
- **Business-key duplicates** — records that share the same flight identity based on flight date, airline, flight number, route, and scheduled departure time.

The business key consists of `FL_DATE`, `OP_UNIQUE_CARRIER`, `OP_CARRIER_FL_NUM`, `ORIGIN`, `DEST`, and `CRS_DEP_TIME`.


In [0]:
# Configure duplicate business key

print("Business key configured successfully.")
print("Business key columns:")
for column_name in cfg.BUSINESS_KEY_COLUMNS:
    print(f"- {column_name}")


#### Exact duplicate analysis

Exact duplicate analysis compares the total number of standardized records with the number of distinct records across all columns. No records are removed during this analysis.


In [0]:
# Analyze exact duplicate records

standardized_row_count = df_clean.count()
distinct_row_count = df_clean.dropDuplicates().count()
exact_duplicate_count = standardized_row_count - distinct_row_count

exact_duplicate_summary = spark.createDataFrame(
    [
        (
            standardized_row_count,
            distinct_row_count,
            exact_duplicate_count,
        )
    ],
    [
        "TOTAL_STANDARDIZED_ROWS",
        "DISTINCT_ROWS",
        "EXACT_DUPLICATE_ROWS",
    ],
)

display(exact_duplicate_summary)


#### Business key duplicate analysis

Business-key duplicate analysis identifies repeated flight identities using the project business key. No records are removed automatically during this stage.


In [0]:
# Analyze business-key duplicate records

business_key_duplicates = (
    df_clean
    .groupBy(*cfg.BUSINESS_KEY_COLUMNS)
    .agg(F.count("*").alias("RECORD_COUNT"))
    .filter(F.col("RECORD_COUNT") > 1)
    .orderBy(F.col("RECORD_COUNT").desc())
)

duplicate_business_key_count = business_key_duplicates.count()

duplicate_summary_row = (
    business_key_duplicates
    .agg(
        F.coalesce(F.sum("RECORD_COUNT"), F.lit(0)).cast("long").alias("TOTAL_DUPLICATE_RECORDS"),
        F.coalesce(F.sum(F.col("RECORD_COUNT") - 1), F.lit(0)).cast("long").alias("ADDITIONAL_RECORDS_BEYOND_FIRST"),
    )
    .first()
)

business_key_summary = spark.createDataFrame(
    [
        (
            int(duplicate_business_key_count),
            int(duplicate_summary_row["TOTAL_DUPLICATE_RECORDS"]),
            int(duplicate_summary_row["ADDITIONAL_RECORDS_BEYOND_FIRST"]),
        )
    ],
    schema="DUPLICATE_BUSINESS_KEYS long, TOTAL_DUPLICATE_RECORDS long, ADDITIONAL_RECORDS_BEYOND_FIRST long",
)

display(business_key_summary)


#### Missing value analysis

Missing values are evaluated across all selected variables to determine whether they represent data-quality issues or expected operational conditions. No records are removed or imputed during this stage.


In [0]:
# Summarize missing values by column

total_rows = df_clean.count()
null_summary_expressions = []

for column_name in df_clean.columns:
    null_summary_expressions.append(
        F.sum(F.when(F.col(column_name).isNull(), 1).otherwise(0)).alias(column_name)
    )

null_counts_row = df_clean.select(*null_summary_expressions).first()

null_summary_rows = []
for column_name in df_clean.columns:
    null_count = int(null_counts_row[column_name])
    null_percentage = (null_count / total_rows) * 100 if total_rows > 0 else 0.0
    null_summary_rows.append((column_name, null_count, round(null_percentage, 4)))

null_summary_df = spark.createDataFrame(
    null_summary_rows,
    schema="COLUMN_NAME string, NULL_COUNT long, NULL_PERCENTAGE double",
)

display(null_summary_df.orderBy(F.col("NULL_PERCENTAGE").desc(), F.col("COLUMN_NAME")))


In [0]:
# Review columns containing null values

display(null_summary_df.filter(F.col("NULL_COUNT") > 0))


In [0]:
# Summarize missing-value coverage

missing_value_overview = (
    null_summary_df
    .agg(
        F.sum(F.when(F.col("NULL_COUNT") > 0, 1).otherwise(0)).alias("COLUMNS_WITH_NULLS"),
        F.sum(F.when(F.col("NULL_COUNT") == 0, 1).otherwise(0)).alias("COLUMNS_WITHOUT_NULLS"),
        F.max("NULL_PERCENTAGE").alias("HIGHEST_NULL_PERCENTAGE"),
    )
)

display(missing_value_overview)


#### Missing value investigation

This section examines the relationship between missing values and the operational status of flights, particularly cancellations and diversions.


In [0]:
# Investigate missing values by flight status

display(
    df_clean.groupBy(cfg.CANCELLED_COLUMN, cfg.DIVERTED_COLUMN).agg(
        F.count("*").alias("TOTAL"),
        F.sum(F.when(F.col(cfg.TARGET_COLUMN).isNull(), 1).otherwise(0)).alias("NULL_TARGET"),
    )
)


#### Missing target investigation

Nearly all missing values in `ARR_DEL15` are associated with cancelled or diverted flights. Records that are not cancelled or diverted but still contain a missing target are reviewed separately.


In [0]:
# Inspect remaining missing target records

remaining_missing_target_record = df_clean.filter(
    (F.col(cfg.CANCELLED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
    & (F.col(cfg.DIVERTED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
    & F.col(cfg.TARGET_COLUMN).isNull()
)

print(
    "Remaining non-cancelled, non-diverted records "
    f"with missing {cfg.TARGET_COLUMN}: {remaining_missing_target_record.count()}"
)

display(remaining_missing_target_record)


#### Target cleaning decision

Records with a missing target variable (`ARR_DEL15`) for completed flights cannot be used for supervised machine learning and are excluded from the cleaned dataset. Remaining missing values associated with cancellations or diversions are retained.


In [0]:
# Remove incomplete target records

invalid_target_records = df_clean.filter(
    (F.col(cfg.CANCELLED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
    & (F.col(cfg.DIVERTED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
    & F.col(cfg.TARGET_COLUMN).isNull()
)

print(f"Invalid target records: {invalid_target_records.count()}")

df_clean = df_clean.filter(
    ~(
        (F.col(cfg.CANCELLED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
        & (F.col(cfg.DIVERTED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
        & F.col(cfg.TARGET_COLUMN).isNull()
    )
)

print(f"Rows after cleaning: {df_clean.count():,}")


#### Domain validation

Domain validation verifies that selected categorical and binary variables contain only values permitted by the BTS data dictionary and the project analytical definitions.


In [0]:
# Validate categorical and binary domains

domain_validation_rows = []

for column_name, valid_values in cfg.DOMAIN_RULES.items():
    invalid_count = (
        df_clean
        .filter(F.col(column_name).isNotNull() & ~F.col(column_name).isin(valid_values))
        .count()
    )
    null_count = df_clean.filter(F.col(column_name).isNull()).count()
    observed_values = [
        row[column_name]
        for row in df_clean.select(column_name).distinct().orderBy(column_name).collect()
    ]
    domain_validation_rows.append(
        (column_name, ", ".join(map(str, valid_values)), str(observed_values), invalid_count, null_count)
    )

domain_validation_df = spark.createDataFrame(
    domain_validation_rows,
    schema="COLUMN_NAME string, EXPECTED_VALUES string, OBSERVED_VALUES string, INVALID_RECORDS long, NULL_RECORDS long",
)

display(domain_validation_df)


#### Business rule validation

Business-rule validation checks whether related variables are logically consistent with one another.

**Departure delay indicator consistency**

The `DEP_DEL15` indicator should agree with the recorded departure delay (`DEP_DELAY`). Records with null values are evaluated separately and are not counted as violations in this check.


In [0]:
# Validate departure delay consistency

departure_delay_rule_summary = df_clean.select(
    F.count("*").alias("TOTAL_ROWS"),
    F.sum(
        F.when(
            (F.col(cfg.DEPARTURE_DELAY_FLAG_COLUMN) == 1)
            & F.col(cfg.DEPARTURE_DELAY_COLUMN).isNotNull()
            & (F.col(cfg.DEPARTURE_DELAY_COLUMN) < cfg.DELAY_THRESHOLD_MINUTES),
            1,
        ).otherwise(0)
    ).alias("INDICATOR_1_BUT_DELAY_BELOW_15"),
    F.sum(
        F.when(
            (F.col(cfg.DEPARTURE_DELAY_FLAG_COLUMN) == 0)
            & F.col(cfg.DEPARTURE_DELAY_COLUMN).isNotNull()
            & (F.col(cfg.DEPARTURE_DELAY_COLUMN) >= cfg.DELAY_THRESHOLD_MINUTES),
            1,
        ).otherwise(0)
    ).alias("INDICATOR_0_BUT_DELAY_AT_LEAST_15"),
    F.sum(
        F.when(
            F.col(cfg.DEPARTURE_DELAY_FLAG_COLUMN).isNull()
            | F.col(cfg.DEPARTURE_DELAY_COLUMN).isNull(),
            1,
        ).otherwise(0)
    ).alias("ROWS_WITH_NULL_DEPARTURE_DELAY_FIELDS"),
)

display(departure_delay_rule_summary)


#### Arrival delay indicator consistency

The `ARR_DEL15` indicator should be logically consistent with the recorded arrival delay (`ARR_DELAY`). Records with null arrival values are evaluated separately because they may correspond to cancelled or diverted flights.


In [0]:
# Validate arrival delay consistency

arrival_delay_rule_summary = df_clean.select(
    F.count("*").alias("TOTAL_ROWS"),
    F.sum(
        F.when(
            (F.col(cfg.ARRIVAL_DELAY_FLAG_COLUMN) == 1)
            & F.col(cfg.ARRIVAL_DELAY_COLUMN).isNotNull()
            & (F.col(cfg.ARRIVAL_DELAY_COLUMN) < cfg.DELAY_THRESHOLD_MINUTES),
            1,
        ).otherwise(0)
    ).alias("INDICATOR_1_BUT_DELAY_BELOW_15"),
    F.sum(
        F.when(
            (F.col(cfg.ARRIVAL_DELAY_FLAG_COLUMN) == 0)
            & F.col(cfg.ARRIVAL_DELAY_COLUMN).isNotNull()
            & (F.col(cfg.ARRIVAL_DELAY_COLUMN) >= cfg.DELAY_THRESHOLD_MINUTES),
            1,
        ).otherwise(0)
    ).alias("INDICATOR_0_BUT_DELAY_AT_LEAST_15"),
    F.sum(
        F.when(
            F.col(cfg.ARRIVAL_DELAY_FLAG_COLUMN).isNull()
            | F.col(cfg.ARRIVAL_DELAY_COLUMN).isNull(),
            1,
        ).otherwise(0)
    ).alias("ROWS_WITH_NULL_ARRIVAL_DELAY_FIELDS"),
)

display(arrival_delay_rule_summary)


#### Cancellation code consistency

Flights marked as cancelled should have an associated cancellation reason recorded in `CANCELLATION_CODE`. Flights that are not cancelled are expected to have a null cancellation code.


In [0]:
# Validate cancellation code consistency

cancellation_rule_summary = df_clean.select(
    F.count("*").alias("TOTAL_ROWS"),
    F.sum(
        F.when(
            (F.col(cfg.CANCELLED_COLUMN) == 1)
            & F.col(cfg.CANCELLATION_CODE_COLUMN).isNull(),
            1,
        ).otherwise(0)
    ).alias("CANCELLED_WITHOUT_REASON"),
    F.sum(
        F.when(
            (F.col(cfg.CANCELLED_COLUMN) == 0)
            & F.col(cfg.CANCELLATION_CODE_COLUMN).isNotNull(),
            1,
        ).otherwise(0)
    ).alias("NOT_CANCELLED_WITH_REASON"),
)

display(cancellation_rule_summary)


#### Diversion consistency

Flights marked as diverted follow a different operational process from normally completed flights. The arrival delay indicator (`ARR_DEL15`) is expected to be unavailable for diverted flights.


In [0]:
# Validate diversion consistency

diversion_rule_summary = df_clean.select(
    F.count("*").alias("TOTAL_ROWS"),
    F.sum(
        F.when(
            (F.col(cfg.DIVERTED_COLUMN) == 1)
            & F.col(cfg.TARGET_COLUMN).isNotNull(),
            1,
        ).otherwise(0)
    ).alias("DIVERTED_WITH_TARGET"),
    F.sum(
        F.when(
            (F.col(cfg.DIVERTED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
            & (F.col(cfg.CANCELLED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
            & F.col(cfg.TARGET_COLUMN).isNull(),
            1,
        ).otherwise(0)
    ).alias("NON_DIVERTED_NON_CANCELLED_WITH_NULL_TARGET"),
)

display(diversion_rule_summary)


#### Delay-cause variable consistency

The BTS delay-cause variables describe delay minutes attributed to specific causes. This validation checks whether non-delayed flights contain positive delay-cause values. No records are modified during this validation.


#### Cleaning completion

The data-cleaning workflow validates schema, data types, duplicates, missing values, domain constraints, and business rules before producing the cleaned dataset.

Incomplete records with missing target values for completed flights are excluded. Remaining missing values that represent legitimate operational conditions are retained.


#### Save clean dataset to processed layer

The cleaned dataset is stored in Delta format within the processed layer of the project data lake. Saving the cleaned dataset separately preserves the original raw dataset and supports reproducibility throughout the analytical pipeline.


In [0]:
# Save the cleaned dataset to the processed layer

(
    df_clean.write
    .format("delta")
    .mode("overwrite")
    .save(cfg.CLEAN_DELTA_PATH)
)

print("Clean dataset saved successfully.")
print(f"Location: {cfg.CLEAN_DELTA_PATH}")
print(f"Total cleaned records: {df_clean.count():,}")


In [0]:
# Validate processed Delta copy

df_processed = spark.read.format("delta").load(cfg.CLEAN_DELTA_PATH)

print(f"Processed records: {df_processed.count():,}")

display(df_processed.limit(10))


#### Save cleaned managed Delta table

The cleaned dataset is saved as a managed Delta table in Unity Catalog so it can be queried through SQL, accessed by downstream notebooks, and governed through catalog permissions.

The original `flights_raw` table remains unchanged.


In [0]:
# Register cleaned dataset as managed Delta table

(
    df_clean.writeTo(cfg.CLEAN_TABLE)
    .using("delta")
    .createOrReplace()
)

print("Cleaned managed Delta table created successfully.")
print(f"Table: {cfg.CLEAN_TABLE}")
print(f"Total records: {spark.table(cfg.CLEAN_TABLE).count():,}")
